# Kombinierte Labels: Random Forest

**Ziel:** Crop und Stage als gemeinsame Klasse `Crop|Stage` vorhersagen und die Hyperparameter ausschließlich per Cross-Validation auf dem gemeinsamen Train-Anteil wählen.

**Ablauf:** `tune()` erzeugt einen lokalen MLflow-Tuning-Run mit einem Unterlauf pro Parameterkombination. Die Validierung bleibt für Schritt 4 unberührt.

In [ ]:
%load_ext autoreload
%autoreload 2

from awp2.experiment import TuneConfig, tune
from awp2.models import make_combined_random_forest

## 1. Hyperparameter suchen

Das Raster testet acht Random-Forest-Konfigurationen mit fünf CV-Folds auf dem Train-Anteil. Entscheidend ist ausschließlich `bacc_combined`; der 30-%-Validierungssplit wird erst nach der Auswahl des besten Kandidaten verwendet.

In [ ]:
tuned = tune(
    make_combined_random_forest(),
    TuneConfig(
        name="combined_rf_search",
        approach="combined",
        description=("Random Forest auf kombinierten Crop/Stage-Labels: "
                     "Tiefe, Blattgröße und Feature-Anteil vergleichen."),
        param_grid={
            "estimator__max_depth": [None, 20],
            "estimator__min_samples_leaf": [1, 3],
            "estimator__max_features": ["sqrt", 0.3],
        },
    ),
)

tuned.candidates.sort_values("rank")

# 1.2 Validieren

In [ ]:
from awp2.experiment import RunConfig, run

result = run(
    tuned.best_model,
    RunConfig(
        name="combined_rf",
        approach="combined",
        description=(
            "Bester Random Forest aus combined_rf_search, "
            "einmal auf der Validierung bewertet."
        ),
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## 2. Analyse

Der kombinierte Ansatz wird mit der dokumentierten Random-Forest-Baseline verglichen. Die folgenden Plots zeigen, ob die Cross-Validation die Validierung realistisch eingeschätzt hat und welche Klassen weiterhin verwechselt werden.

In [ ]:
import pandas as pd
from IPython.display import display

from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

BASELINE_BACC_COMBINED = 0.743

comparison = pd.DataFrame(
    {
        "BAcc kombiniert": {
            "Baseline": BASELINE_BACC_COMBINED,
            "Kombinierte Labels": result.metrics.bacc_combined,
        },
        "Delta zur Baseline": {
            "Baseline": 0.0,
            "Kombinierte Labels": result.metrics.bacc_combined - BASELINE_BACC_COMBINED,
        },
    }
)
display(comparison)

ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__max_features",
    validation_score=result.metrics.bacc_combined,
 )
ax.set_title("Kombinierte Labels: CV-Score je Kandidat")
display(ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

In [ ]:
{
    "tuning_run": tuned.run_id,
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "evaluation_run": result.run_id,
}

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__max_features",
    validation_score=result.metrics.bacc_combined,
 )
tuning_ax.set_title("Kombinierte Labels: CV-Score je Kandidat")
save_doc_figure(tuning_ax.figure, "combined_rf_tuning", MODEL_DOCS_FIGURES_DIR)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(confusion_figure, "combined_rf_confusion", MODEL_DOCS_FIGURES_DIR)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(recall_figure, "combined_rf_recall", MODEL_DOCS_FIGURES_DIR)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(confusions_figure, "combined_rf_confusions", MODEL_DOCS_FIGURES_DIR)